In [99]:
import pandas as pd
import json

# carregando json
with open('../dados/dados_nivel_1.json', 'r', encoding='utf-8') as f:
    dados_brutos = json.load(f)

taxa_cambio = dados_brutos['taxa_cambio_usd_brl']
df = pd.DataFrame(dados_brutos['operacoes'])

print(f"Taxa de câmbio USD/BRL: {taxa_cambio}")
print(f"Total de registros carregados: {len(df)}")
df.head()

Taxa de câmbio USD/BRL: 5.4
Total de registros carregados: 20


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [100]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   id           20 non-null     str  
 1   cliente_id   20 non-null     str  
 2   data         19 non-null     str  
 3   valor        20 non-null     int64
 4   moeda        20 non-null     str  
 5   canal        20 non-null     str  
 6   tipo         20 non-null     str  
 7   contraparte  20 non-null     str  
 8   observacao   20 non-null     str  
dtypes: int64(1), str(8)
memory usage: 1.5 KB


In [101]:
df.isnull().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [102]:
df.duplicated().sum()

np.int64(1)

In [103]:
print("Moedas:", df['moeda'].unique())
print("Canais:", df['canal'].unique())
print("Tipos:", df['tipo'].unique())

Moedas: <StringArray>
['BRL', 'USD']
Length: 2, dtype: str
Canais: <StringArray>
['pix', 'ted', 'boleto', 'cartao', 'especie']
Length: 5, dtype: str
Tipos: <StringArray>
['transferencia_enviada', 'pagamento', 'transferencia_recebida', 'deposito']
Length: 4, dtype: str


## Problemas encontrados:

1. Data nula (1 registro) - "OP-0017" veio sem data, com a observação "data não capturada pelo sistema". Como a Regra 1 depende de agrupar por data, esse registro não pode participar dela sem uma decisão explícita.

2. Linha duplicada (1 registro) - "OP-0017" aparece duas vezes, com todos os campos idênticos.

3. Moeda mista (BRL e USD) - a maioria dos valores está em BRL, mas "OP-0013" veio em USD. Precisa ser convertido para BRL usando a taxa fornecida antes de entrar em qualquer soma ou comparação.

In [104]:
# Removendo a linha duplicada, mantendo a primeira ocorrência.
print(f"Linhas antes: {len(df)}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Linhas depois: {len(df)}")

Linhas antes: 20
Linhas depois: 19


In [105]:
# Não removi nem inventei data. Crio uma flag explícita para o registro com data ausente, preservando o dado original (NaN continua NaN).
df['data_ausente'] = df['data'].isnull()

df[df['data_ausente']]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,data_ausente
16,OP-0017,CLI-A-5,NaN,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema,True


In [106]:
# Crio a coluna valor_brl: valores já em BRL ficam como estão;
# Valores em USD são multiplicados pela taxa de câmbio fornecida no JSON.
df['valor_brl'] = df.apply(lambda row: row['valor'] * taxa_cambio if row['moeda'] == 'USD' else row['valor'], axis=1)

# Conferência: mostra apenas os registros que passaram pela conversão.
df[df['moeda'] == 'USD'][['id', 'cliente_id', 'valor', 'moeda', 'valor_brl']]

,id,cliente_id,valor,moeda,valor_brl
12,OP-0013,CLI-A-4,12000,USD,64800.0


In [107]:
# Agregação: volume total por cliente
volume_por_cliente = df.groupby('cliente_id')['valor_brl'].sum().sort_values(ascending=False)
volume_por_cliente

cliente_id
CLI-A-4    79500.0
CLI-A-1    57500.0
CLI-A-2    52900.0
CLI-A-3    48500.0
CLI-A-5    16900.0
CLI-A-6    10200.0
Name: valor_brl, dtype: float64

In [108]:
# Agregação: quantidade de operações por canal
qtd_por_canal = df['canal'].value_counts()
qtd_por_canal

canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

In [109]:
# Regra 1: Fracionamento
# Sinaliza o cliente que em uma mesma data fez 3+ operações cuja soma ultrapassa R$50.000, sendo que nenhuma operação isolada atinge R$20.000.

# Agrupa por cliente + data
agrupado = df.groupby(['cliente_id', 'data']).agg(
    qtd_operacoes=('valor_brl', 'count'),
    soma_valor=('valor_brl', 'sum'),
    maior_operacao=('valor_brl', 'max')
).reset_index()

# Aplica as três condições da regra
agrupado['fracionamento'] = (
    (agrupado['qtd_operacoes'] >= 3) &
    (agrupado['soma_valor'] > 50000) &
    (agrupado['maior_operacao'] < 20000)
)

# Lista de clientes sinalizados
clientes_fracionamento = agrupado[agrupado['fracionamento']]['cliente_id'].unique()
print("Clientes sinalizados pela Regra 1:", clientes_fracionamento)

# Adiciona a flag de volta ao DataFrame original, por cliente
df['flag_fracionamento'] = df['cliente_id'].isin(clientes_fracionamento)

agrupado[agrupado['fracionamento']]

Clientes sinalizados pela Regra 1: <StringArray>
['CLI-A-1']
Length: 1, dtype: str


,cliente_id,data,qtd_operacoes,soma_valor,maior_operacao,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True


In [110]:
# Validação: mostra o par CLI-A-1 (deve disparar) vs CLI-A-2 (não deve disparar)
# Mesmo com volume comparável, para provar que a regra distingue corretamente.

validacao = agrupado[agrupado['cliente_id'].isin(['CLI-A-1', 'CLI-A-2'])]
validacao

,cliente_id,data,qtd_operacoes,soma_valor,maior_operacao,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
1,CLI-A-1,2026-03-21,1,3300.0,3300.0,False
2,CLI-A-2,2026-03-14,2,52900.0,27000.0,False


In [111]:
# Regra 2 - Valor atípico
# Sinaliza a operação cujo valor em BRL seja > 5x a mediana dos valores daquele mesmo cliente. Só se aplica a clientes com 4+ operações.

# Calcula mediana e contagem de operações por cliente
estatisticas_cliente = df.groupby('cliente_id')['valor_brl'].agg(
    mediana_cliente='median',
    qtd_operacoes_cliente='count'
).reset_index()

# Junta essas estatísticas de volta em cada linha do DataFrame original
df = df.merge(estatisticas_cliente, on='cliente_id', how='left')

# Aplica as duas condições da regra
df['flag_valor_atipico'] = (
    (df['qtd_operacoes_cliente'] >= 4) &
    (df['valor_brl'] > 5 * df['mediana_cliente'])
)

# Mostra as operações sinalizadas
df[df['flag_valor_atipico']][['id', 'cliente_id', 'valor_brl', 'mediana_cliente', 'qtd_operacoes_cliente']]

,id,cliente_id,valor_brl,mediana_cliente,qtd_operacoes_cliente
12,OP-0013,CLI-A-4,64800.0,5450.0,4


# Problemas de qualidade encontrados e tratamento

Inspecionei o DataFrame com `.info()`, `.isnull().sum()`, `.duplicated().sum()` 
e `.unique()` nas colunas categóricas. Três problemas foram encontrados:

### 1. Registro duplicado:
"OP-0007" (CLI-A-3) aparecia duas vezes, com todos os campos idênticos. Como o dataset veio de um sistema legado, tratei como duplicação de importação e removi a cópia com `drop_duplicates()`, mantendo a primeira ocorrência. O próprio `id` já indica que é o mesmo registro.

### 2. Data ausente:
"OP-0017" (CLI-A-5) veio com `data: null` e uma observação do próprio sistema ("data não capturada pelo sistema"). Optei por não remover nem deduzir essa data: em contexto de PLD, descartar um registro incompleto pode significar perder justamente o tipo de anomalia que merece investigação. Em vez disso, criei uma coluna booleana `data_ausente` para sinalizar o registro explicitamente. Na prática, isso significa que o "OP-0017" participa normalmente das agregações e da Regra 2(que não depende de data), mas naturalmente não entra em nenhum agrupamento da Regra 1.

### 3. Moeda mista:
A maioria dos valores está em BRL, mas "OP-0013" (CLI-A-4) veio em USD. Converti para BRL usando a taxa fornecida no próprio arquivo, criando a coluna `valor_brl`, usada em todas as agregações e regras a partir daqui. Sem essa conversão, o volume desse cliente apareceria menor do que o real.

In [112]:
import os
import time
import json
from dotenv import load_dotenv
from google import genai

load_dotenv('../.env')

client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))

print("Cliente Gemini configurado.")

Cliente Gemini configurado.


In [113]:
# Monta o recorte de dados do cliente escolhido para análise da LLM.
# Trazemos as colunas relevantes para parecer (id, data, valor já em BRL, moeda original, canal, tipo, contraparte) 
# e a flag que motivou a sinalização, para que o prompt possa referenciar o motivo do alerta.
cliente_analise = 'CLI-A-4'

operacoes_cliente = df[df['cliente_id'] == cliente_analise][
    ['id', 'data', 'valor_brl', 'moeda', 'canal', 'tipo', 'contraparte', 'flag_valor_atipico']
]

operacoes_cliente

,id,data,valor_brl,moeda,canal,tipo,contraparte,flag_valor_atipico
9,OP-0010,2026-03-03,3800.0,BRL,cartao,pagamento,Alfa Comercio LTDA,False
10,OP-0011,2026-03-11,5100.0,BRL,boleto,pagamento,Beta Servicos ME,False
11,OP-0012,2026-03-18,5800.0,BRL,pix,transferencia_enviada,Gama Distribuidora,False
12,OP-0013,2026-03-24,64800.0,USD,ted,transferencia_recebida,Zeta Importacao,True


In [114]:
# Monta o resumo textual das operações do cliente para incluir no prompt.
resumo_operacoes = operacoes_cliente.to_string(index=False)

prompt_v1 = f"""Você é um analista de prevenção à lavagem de dinheiro (PLD) 
de um banco. Analise as operações abaixo do cliente {cliente_analise}, que 
foi sinalizado por uma regra determinística de valor atípico (operação com 
valor muito acima do padrão histórico do próprio cliente).

Operações do cliente:
{resumo_operacoes}

Responda APENAS com um JSON válido, sem texto adicional antes ou depois, 
no seguinte formato:
{{
  "nivel_risco": "baixo" | "medio" | "alto",
  "tipologia_suspeita": "string descrevendo a tipologia, ou 'nenhuma identificada'",
  "red_flags": ["lista", "de", "strings"],
  "justificativa": "string explicando o raciocínio"
}}
"""

# Chamada à API, com registro de tempo de resposta
inicio = time.time()
resposta = client.models.generate_content(
    model='gemini-3.6-flash',
    contents=prompt_v1
)
tempo_resposta = time.time() - inicio

print(f"Tempo de resposta: {tempo_resposta:.2f}s")
print(resposta.text)

Tempo de resposta: 37.26s
{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Movimentação incompatível com o perfil patrimonial/financeiro / Suspeita de Lavagem de Dinheiro baseada em Comércio Exterior",
  "red_flags": [
    "Entrada de recurso de valor significativamente superior ao padrão histórico do cliente",
    "Movimentação em moeda estrangeira (USD) totalmente discrepante do histórico prévio em BRL",
    "Mudança abrupta e injustificada no padrão comportamental de transações",
    "Contraparte com perfil corporativo de importação (Zeta Importacao) sem histórico de relacionamento prévio"
  ],
  "justificativa": "O cliente possui histórico de movimentações baixas e domésticas em BRL (entre R$ 3.800 e R$ 5.800). A operação OP-0013 representa um desvio drástico de padrão ao receber USD 64.800,00 de uma empresa de importação. A magnitude do valor, aliada ao uso de moeda estrangeira e à ausência de histórico que justifique essa atividade comercial/financeira, eleva o risco de lavage

In [115]:
def parse_resposta_llm(texto_resposta):
    """Tenta extrair um JSON válido da resposta da LLM.
    LLMs às vezes envolvem o JSON em blocos markdown (```json ... ```) 
    mesmo quando instruídas a não fazer isso — tratamos esse caso."""
    texto_limpo = texto_resposta.strip()
    
    # Remove blocos de código markdown, se existirem
    if texto_limpo.startswith('```'):
        texto_limpo = texto_limpo.split('```')[1]
        if texto_limpo.startswith('json'):
            texto_limpo = texto_limpo[4:]
        texto_limpo = texto_limpo.strip()
    
    try:
        parecer = json.loads(texto_limpo)
        campos_esperados = {'nivel_risco', 'tipologia_suspeita', 'red_flags', 'justificativa'}
        if not campos_esperados.issubset(parecer.keys()):
            faltando = campos_esperados - parecer.keys()
            return {'erro': f'Campos ausentes na resposta: {faltando}', 'resposta_bruta': texto_resposta}
        return parecer
    except json.JSONDecodeError as e:
        return {'erro': f'JSON malformado: {e}', 'resposta_bruta': texto_resposta}

parecer_v1 = parse_resposta_llm(resposta.text)
parecer_v1

{'nivel_risco': 'alto',
 'tipologia_suspeita': 'Movimentação incompatível com o perfil patrimonial/financeiro / Suspeita de Lavagem de Dinheiro baseada em Comércio Exterior',
 'red_flags': ['Entrada de recurso de valor significativamente superior ao padrão histórico do cliente',
  'Movimentação em moeda estrangeira (USD) totalmente discrepante do histórico prévio em BRL',
  'Mudança abrupta e injustificada no padrão comportamental de transações',
  'Contraparte com perfil corporativo de importação (Zeta Importacao) sem histórico de relacionamento prévio'],
 'justificativa': 'O cliente possui histórico de movimentações baixas e domésticas em BRL (entre R$ 3.800 e R$ 5.800). A operação OP-0013 representa um desvio drástico de padrão ao receber USD 64.800,00 de uma empresa de importação. A magnitude do valor, aliada ao uso de moeda estrangeira e à ausência de histórico que justifique essa atividade comercial/financeira, eleva o risco de lavagem de dinheiro e exige apuração aprofundada/com